In [ ]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier, log_evaluation, early_stopping
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import optuna
import warnings
warnings.filterwarnings('ignore')
# Load the dataset
data_path = '../data/processed/'
data      = pd.read_parquet(data_path + 'train_data.parquet')
data.head()

In [ ]:
#baseline model
X = data.drop(columns=['TARGET', 'SK_ID_CURR'])
y = data['TARGET']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=42
    )
model.fit(X_train, y_train) 

In [ ]:
y_pred = model.predict(X_val)
print(classification_report(y_val, y_pred))
print(confusion_matrix(y_val, y_pred))
y_pred_proba = model.predict_proba(X_val)[:, 1]
print(f'ROC AUC Score: {roc_auc_score(y_val, y_pred_proba)}')

In [ ]:
# import time
# from tqdm import tqdm
# def objective(trial):
#     params = {
#         'objective': 'binary',
#         'metric': 'auc',
#         'n_jobs': -1,
#         'random_state': 42,
#         'n_estimators': 10000, 
#         'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        
#         # --- KIỂM SOÁT CẤU TRÚC CÂY ---
#         'num_leaves': trial.suggest_int('num_leaves', 20, 150),
#         'max_depth': trial.suggest_int('max_depth', 3, 12),
#         'min_child_samples': trial.suggest_int('min_child_samples', 100, 2000), 
        
#         # --- LẤY MẪU NGẪU NHIÊN ---
#         'subsample': trial.suggest_float('subsample', 0.5, 1.0),
#         'subsample_freq': trial.suggest_int('subsample_freq', 1, 10), 
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        
#         # --- REGULARIZATION ---
#         'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),  
#         'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True), 
        
#         # --- XỬ LÝ MẤT CÂN BẰNG DỮ LIỆU ---
#         'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 15.0) 
#     }
#     start_trial_time = time.time()
    
#     model = LGBMClassifier(**params)
#     model.fit(
#         X_train, y_train,
#         eval_set=[(X_val, y_val)],
#         callbacks=[
#             early_stopping(stopping_rounds=100, verbose=False),
#             log_evaluation(0)
#         ]
#     )
    
#     y_pred_proba    = model.predict_proba(X_val)[:, 1]
#     auc_score       = roc_auc_score(y_val, y_pred_proba)
#     trial_time      = time.time() - start_trial_time
#     tqdm.write(f"Trial {trial.number} | Time: {trial_time:.1f}s | Iterations: {model.best_iteration_} | AUC: {auc_score:.4f}")
    
#     return auc_score

# warnings.filterwarnings('ignore')
# n_trials = 50
# pbar = tqdm(total=n_trials, desc="🚀 Đang Tuning LightGBM")
# def optuna_callback(study, trial):
#     pbar.update(1)
#     pbar.set_postfix({'Best AUC': f"{study.best_value:.4f}"})
# total_start_time = time.time()

# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=n_trials, callbacks=[optuna_callback])

# pbar.close()
# total_time_mins = (time.time() - total_start_time) / 60

# print("="*50)
# print(f"Hoàn thành Tuning trong {total_time_mins:.2f} phút!")
# print(f"Best AUC: {study.best_value:.4f}")
# print("Best Params:")
# for key, value in study.best_params.items():
#     print(f"    '{key}': {value},")

In [ ]:
# Hoàn thành Tuning trong 143.29 phút!
# Best AUC: 0.7903
# Best Params:
#     'learning_rate': 0.007067006815027388,
#     'num_leaves': 138,
#     'max_depth': 7,
#     'min_child_samples': 1512,
#     'subsample': 0.8820047422857376,
#     'subsample_freq': 8,
#     'colsample_bytree': 0.40093134941872577,
#     'reg_alpha': 0.4544560417169174,
#     'reg_lambda': 0.0018128371472283505,
#     'scale_pos_weight': 1.906842313442258,
# study.best_params
best_params = {
    'learning_rate': 0.007067006815027388,
    'num_leaves': 138,
    'max_depth': 7,
    'min_child_samples': 1512,
    'subsample': 0.8820047422857376,
    'subsample_freq': 8,
    'colsample_bytree': 0.40093134941872577,
    'reg_alpha': 0.4544560417169174,
    'reg_lambda': 0.0018128371472283505,
    'scale_pos_weight': 1.906842313442258
}

In [ ]:
best_model = LGBMClassifier(n_estimators=10000, **best_params)
best_model.fit(X_train, y_train)
y_train_pred = best_model.predict_proba(X_train)[:, 1]
y_pred_proba = best_model.predict_proba(X_val)[:, 1]
print(f"ROC AUC Score train: {roc_auc_score(y_train, y_train_pred)}")
print(f'ROC AUC Score: {roc_auc_score(y_val, y_pred_proba)}')